# Calendar & Promotion Feature Engineering

## Business Objective

Evaluate whether calendar characteristics and historical promotional activity provide useful information for forecasting total monthly retail sales.

## Forecasting Design

- **Target:** Monthly Total Sales
- **Historical period:** January 2013–July 2017
- **Forecast horizon:** 3 months
- **Minimum training history:** 36 months
- **Validation:** 17 expanding rolling-origin windows
- **Forecast cases:** 51
- **Existing benchmark:** 50/50 Blended model — 8.36% WAPE
- **Alternative benchmark:** ETS Additive — 8.48% WAPE

## Data Governance Rules

1. Do not modify raw or standardized source datasets.
2. Preserve the monthly forecasting grain.
3. Use only information available at the relevant forecast origin.
4. Do not use realized future promotions as predictors.
5. Validate features before modeling.
6. Compare models on identical historical forecast cases.

## Step 1 — Feature Availability Audit

We will first inspect the existing analytical datasets and identify suitable calendar and promotion variables.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "processed").exists()
)

print("Project root:", PROJECT_ROOT)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

required_files = [
    "data/processed/monthly_sales_complete.csv",
    "data/processed/train_standardized.csv",
    "reports/retail_driver_analysis/monthly_sales_promotion_performance.csv",
    "reports/forecasting_feature_evaluation/seven_model_matched_forecasts.csv",
]

for relative_path in required_files:
    file_path = PROJECT_ROOT / relative_path
    print(
        "FOUND" if file_path.is_file() else "MISSING",
        relative_path
    )

assert all(
    (PROJECT_ROOT / path).is_file()
    for path in required_files
)

print("Environment and input file validation: PASSED")

Project root: d:\GitHub\retail-sales-forecasting
Pandas: 3.0.6
NumPy: 2.5.3
FOUND data/processed/monthly_sales_complete.csv
FOUND data/processed/train_standardized.csv
FOUND reports/retail_driver_analysis/monthly_sales_promotion_performance.csv
FOUND reports/forecasting_feature_evaluation/seven_model_matched_forecasts.csv
Environment and input file validation: PASSED


In [2]:
# Load existing analytical datasets

monthly_sales_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/monthly_sales_complete.csv"
)

monthly_promotion_df = pd.read_csv(
    PROJECT_ROOT
    / "reports/retail_driver_analysis/monthly_sales_promotion_performance.csv"
)

# Inspect dataset structure
print("MONTHLY SALES DATASET")
print("Rows:", len(monthly_sales_df))
print("Columns:", monthly_sales_df.columns.tolist())
print("\nData types:")
print(monthly_sales_df.dtypes)

print("\n" + "=" * 50)

print("MONTHLY PROMOTION DATASET")
print("Rows:", len(monthly_promotion_df))
print("Columns:", monthly_promotion_df.columns.tolist())
print("\nData types:")
print(monthly_promotion_df.dtypes)

MONTHLY SALES DATASET
Rows: 55
Columns: ['month', 'total_sales', 'source_rows', 'sales_dates_observed', 'stores_observed', 'families_observed', 'calendar_days', 'sales_closure_days', 'is_partial_month', 'sales_coverage_complete', 'is_forecast_training_period']

Data types:
month                              str
total_sales                    float64
source_rows                      int64
sales_dates_observed             int64
stores_observed                  int64
families_observed                int64
calendar_days                    int64
sales_closure_days               int64
is_partial_month                  bool
sales_coverage_complete           bool
is_forecast_training_period       bool
dtype: object

MONTHLY PROMOTION DATASET
Rows: 55
Columns: ['month', 'monthly_sales', 'total_records', 'promotional_records', 'total_promoted_item_counts', 'promotion_record_share_pct', 'avg_sales_per_record']

Data types:
month                             str
monthly_sales                 float6

In [3]:
# Convert month columns to datetime

monthly_sales_df["month"] = pd.to_datetime(
    monthly_sales_df["month"]
)

monthly_promotion_df["month"] = pd.to_datetime(
    monthly_promotion_df["month"]
)

# Validate monthly keys
assert monthly_sales_df["month"].notna().all()
assert monthly_promotion_df["month"].notna().all()

assert monthly_sales_df["month"].is_unique
assert monthly_promotion_df["month"].is_unique

assert monthly_sales_df["month"].dt.is_month_start.all()
assert monthly_promotion_df["month"].dt.is_month_start.all()

# Compare month coverage
sales_months = set(monthly_sales_df["month"])
promotion_months = set(monthly_promotion_df["month"])

assert sales_months == promotion_months

# Reconcile monthly sales values
reconciliation_df = monthly_sales_df[
    ["month", "total_sales"]
].merge(
    monthly_promotion_df[
        ["month", "monthly_sales"]
    ],
    on="month",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert (reconciliation_df["_merge"] == "both").all()

reconciliation_df["sales_difference"] = (
    reconciliation_df["total_sales"]
    - reconciliation_df["monthly_sales"]
)

max_difference = (
    reconciliation_df["sales_difference"].abs().max()
)

assert max_difference < 0.01

print("Monthly key validation: PASSED")
print("Sales reconciliation: PASSED")
print("Matched months:", len(reconciliation_df))
print("Maximum sales difference:", round(max_difference, 6))
print(
    "Date range:",
    reconciliation_df["month"].min().date(),
    "to",
    reconciliation_df["month"].max().date()
)

Monthly key validation: PASSED
Sales reconciliation: PASSED
Matched months: 55
Maximum sales difference: 0.0
Date range: 2013-01-01 to 2017-07-01


In [4]:
# Select the relevant promotion columns
promotion_features = monthly_promotion_df[
    [
        "month",
        "total_records",
        "promotional_records",
        "total_promoted_item_counts",
        "promotion_record_share_pct",
    ]
].copy()

# Merge with monthly sales
monthly_features_df = monthly_sales_df.merge(
    promotion_features,
    on="month",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Validate the merge
assert len(monthly_features_df) == 55
assert (monthly_features_df["_merge"] == "both").all()
assert monthly_features_df["month"].is_unique

monthly_features_df = (
    monthly_features_df
    .drop(columns="_merge")
    .sort_values("month")
    .reset_index(drop=True)
)

# Check missing values in selected columns
selected_columns = [
    "month",
    "total_sales",
    "calendar_days",
    "total_records",
    "promotional_records",
    "total_promoted_item_counts",
    "promotion_record_share_pct",
]

assert not monthly_features_df[
    selected_columns
].isna().any().any()

print("Monthly feature merge validation: PASSED")
print("Rows:", len(monthly_features_df))
print("Columns:", len(monthly_features_df.columns))

display(monthly_features_df[selected_columns].head())

Monthly feature merge validation: PASSED
Rows: 55
Columns: 15


,month,total_sales,calendar_days,total_records,promotional_records,total_promoted_item_counts,promotion_record_share_pct
0,2013-01-01,1.032762e+07,31,55242,0,0,0.0
1,2013-02-01,9.658960e+06,28,49896,0,0,0.0
2,2013-03-01,1.142850e+07,31,55242,0,0,0.0
3,2013-04-01,1.099346e+07,30,53460,0,0,0.0
4,2013-05-01,1.159770e+07,31,55242,0,0,0.0


## Step 2 — Forecast-Time Feature Availability Audit

### Forecasting scenario

At the end of April 2017, predict monthly sales for May, June, and July 2017.

### Known-in-advance features

- Calendar month
- Calendar year
- Days in month
- Calendar quarter

### Historical-only features

- Previous observed monthly sales
- Historical promotional record share
- Historical promoted-item counts
- Historical store coverage

Historical values must be restricted to observations available at the forecast origin.

### Features requiring caution

- `sales_closure_days`: May describe observed closures rather than advance-known closures.
- `stores_observed`: Historical operational coverage, not a guaranteed future store count.
- `total_records`: Actual future transaction/source-record counts are unknown.
- `promotional_records`: Actual future promotion counts are unknown.
- `promotion_record_share_pct`: Actual future promotion share is unknown.
- `total_promoted_item_counts`: Actual future promoted-item counts are unknown.

### Data leakage prevention

Do not use realized values from May, June, or July 2017 when generating forecasts from an April 2017 origin.

### Promotion data quality question

Investigate the first and last months with recorded promotions. A zero may represent no promotions or a source-data limitation; the current dataset alone does not establish which explanation is correct.

In [5]:
# Investigate monthly promotion coverage

promotion_audit_df = monthly_features_df[
    [
        "month",
        "total_records",
        "promotional_records",
        "total_promoted_item_counts",
        "promotion_record_share_pct",
    ]
].copy()

promotion_audit_df["has_recorded_promotions"] = (
    promotion_audit_df["promotional_records"] > 0
)

positive_months = promotion_audit_df[
    promotion_audit_df["has_recorded_promotions"]
]

zero_months = promotion_audit_df[
    ~promotion_audit_df["has_recorded_promotions"]
]

assert (
    promotion_audit_df["promotional_records"]
    <= promotion_audit_df["total_records"]
).all()

assert (
    promotion_audit_df[
        ["total_records", "promotional_records",
         "total_promoted_item_counts"]
    ] >= 0
).all().all()

assert promotion_audit_df[
    "promotion_record_share_pct"
].between(0, 100).all()

print("Promotion coverage audit: PASSED")
print("Total months:", len(promotion_audit_df))
print("Months with recorded promotions:", len(positive_months))
print("Months without recorded promotions:", len(zero_months))

if not positive_months.empty:
    print(
        "First recorded promotion month:",
        positive_months["month"].min().date()
    )
    print(
        "Last recorded promotion month:",
        positive_months["month"].max().date()
    )

print("\nFirst 15 months:")
display(promotion_audit_df.head(15))

Promotion coverage audit: PASSED
Total months: 55
Months with recorded promotions: 40
Months without recorded promotions: 15
First recorded promotion month: 2014-04-01
Last recorded promotion month: 2017-07-01

First 15 months:


,month,total_records,promotional_records,total_promoted_item_counts,promotion_record_share_pct,has_recorded_promotions
0,2013-01-01,55242,0,0,0.0,False
1,2013-02-01,49896,0,0,0.0,False
2,2013-03-01,55242,0,0,0.0,False
3,2013-04-01,53460,0,0,0.0,False
4,2013-05-01,55242,0,0,0.0,False
5,2013-06-01,53460,0,0,0.0,False
6,2013-07-01,55242,0,0,0.0,False
7,2013-08-01,55242,0,0,0.0,False
8,2013-09-01,53460,0,0,0.0,False
9,2013-10-01,55242,0,0,0.0,False


In [6]:
# Identify the first month with recorded promotions
first_promotion_month = promotion_audit_df.loc[
    promotion_audit_df["has_recorded_promotions"],
    "month"
].min()

# Create a conservative promotion coverage flag
monthly_features_df["promotion_coverage_status"] = np.where(
    monthly_features_df["month"] < first_promotion_month,
    "COVERAGE_UNCERTAIN",
    "RECORDED_PERIOD"
)

# Validation
assert first_promotion_month == pd.Timestamp("2014-04-01")

assert (
    monthly_features_df["promotion_coverage_status"]
    .value_counts()
    .to_dict()
    == {
        "RECORDED_PERIOD": 40,
        "COVERAGE_UNCERTAIN": 15,
    }
)

assert monthly_features_df["month"].is_unique
assert len(monthly_features_df) == 55

print("Promotion coverage classification: PASSED")
print("First recorded promotion:", first_promotion_month.date())

print("\nCoverage classification:")
print(
    monthly_features_df[
        "promotion_coverage_status"
    ].value_counts()
)

display(
    monthly_features_df.loc[
        monthly_features_df["month"].between(
            "2014-02-01", "2014-06-01"
        ),
        [
            "month",
            "promotional_records",
            "promotion_record_share_pct",
            "promotion_coverage_status",
        ]
    ]
)

Promotion coverage classification: PASSED
First recorded promotion: 2014-04-01

Coverage classification:
promotion_coverage_status
RECORDED_PERIOD       40
COVERAGE_UNCERTAIN    15
Name: count, dtype: int64


,month,promotional_records,promotion_record_share_pct,promotion_coverage_status
13,2014-02-01,0,0.000000,COVERAGE_UNCERTAIN
14,2014-03-01,0,0.000000,COVERAGE_UNCERTAIN
15,2014-04-01,399,0.746352,RECORDED_PERIOD
16,2014-05-01,1964,3.555266,RECORDED_PERIOD
17,2014-06-01,4367,8.168724,RECORDED_PERIOD


## Step 3 — Forecast-Time Feature Availability Decisions

The feature audit separates information known in advance,
historical information available at the forecast origin,
and realized information that cannot be used for future forecasts.

Promotion coverage is uncertain before April 2014.

Historical promotion variables will require origin-aware
feature engineering. No future realized promotion measures
will be used as predictors.

In [7]:
feature_audit_records = [
    ("month", "KNOWN_IN_ADVANCE", "Calendar month is known"),
    ("calendar_days", "KNOWN_IN_ADVANCE", "Calendar days are known"),
    ("total_sales", "TARGET", "Future sales must not be used"),
    ("promotion_record_share_pct", "HISTORICAL_ONLY",
     "Use only observations available at forecast origin"),
    ("total_promoted_item_counts", "HISTORICAL_ONLY",
     "Use only observations available at forecast origin"),
    ("promotional_records", "HISTORICAL_ONLY",
     "Future realized counts are unavailable"),
    ("sales_closure_days", "REQUIRES_INVESTIGATION",
     "Confirm whether closure dates were known in advance"),
    ("stores_observed", "HISTORICAL_ONLY",
     "Future observed store coverage is unknown"),
    ("total_records", "HISTORICAL_ONLY",
     "Future realized source-record counts are unknown"),
    ("promotion_coverage_status", "QUALITY_FLAG",
     "Early promotion coverage is uncertain"),
]

feature_audit_df = pd.DataFrame(
    feature_audit_records,
    columns=["feature", "availability", "reason"]
)

assert feature_audit_df["feature"].is_unique
assert feature_audit_df["feature"].isin(
    monthly_features_df.columns
).all()

assert len(feature_audit_df) == 10

print("Feature availability audit validation: PASSED")
print("Features audited:", len(feature_audit_df))

display(feature_audit_df)

Feature availability audit validation: PASSED
Features audited: 10


,feature,availability,reason
0,month,KNOWN_IN_ADVANCE,Calendar month is known
1,calendar_days,KNOWN_IN_ADVANCE,Calendar days are known
2,total_sales,TARGET,Future sales must not be used
3,promotion_record_share_pct,HISTORICAL_ONLY,Use only observations available at forecast or...
4,total_promoted_item_counts,HISTORICAL_ONLY,Use only observations available at forecast or...
5,promotional_records,HISTORICAL_ONLY,Future realized counts are unavailable
6,sales_closure_days,REQUIRES_INVESTIGATION,Confirm whether closure dates were known in ad...
7,stores_observed,HISTORICAL_ONLY,Future observed store coverage is unknown
8,total_records,HISTORICAL_ONLY,Future realized source-record counts are unknown
9,promotion_coverage_status,QUALITY_FLAG,Early promotion coverage is uncertain


In [8]:
# Create the report directory
report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_engineering"
)

report_dir.mkdir(parents=True, exist_ok=True)

# Export feature availability decisions
audit_path = report_dir / "feature_availability_audit.csv"

feature_audit_df.to_csv(
    audit_path,
    index=False
)

# Read the report back for validation
saved_audit_df = pd.read_csv(audit_path)

assert len(saved_audit_df) == 10
assert saved_audit_df["feature"].is_unique

pd.testing.assert_frame_equal(
    feature_audit_df.reset_index(drop=True),
    saved_audit_df.reset_index(drop=True)
)

print("Feature availability audit export: PASSED")
print("Report:", audit_path.relative_to(PROJECT_ROOT))
print("Features exported:", len(saved_audit_df))

Feature availability audit export: PASSED
Report: reports\forecasting_feature_engineering\feature_availability_audit.csv
Features exported: 10


## Step 4 — Calendar Feature Engineering

Create calendar features that are known in advance
and can be safely used for future monthly forecasting.

These features describe the forecast month itself.
They do not use future actual sales or promotion observations.

In [9]:
# Create advance-known calendar features
monthly_features_df["calendar_year"] = (
    monthly_features_df["month"].dt.year
)

monthly_features_df["calendar_month"] = (
    monthly_features_df["month"].dt.month
)

monthly_features_df["calendar_quarter"] = (
    monthly_features_df["month"].dt.quarter
)

monthly_features_df["days_in_month"] = (
    monthly_features_df["month"].dt.days_in_month
)

monthly_features_df["is_december"] = (
    monthly_features_df["calendar_month"] == 12
).astype(int)

monthly_features_df["is_february"] = (
    monthly_features_df["calendar_month"] == 2
).astype(int)

# Validate calendar features
assert len(monthly_features_df) == 55
assert monthly_features_df["month"].is_unique

assert monthly_features_df["calendar_month"].between(1, 12).all()
assert monthly_features_df["calendar_quarter"].between(1, 4).all()
assert monthly_features_df["days_in_month"].between(28, 31).all()

assert (
    monthly_features_df["days_in_month"]
    == monthly_features_df["calendar_days"]
).all()

assert monthly_features_df["is_december"].isin([0, 1]).all()
assert monthly_features_df["is_february"].isin([0, 1]).all()

print("Calendar feature engineering: PASSED")
print("Monthly observations:", len(monthly_features_df))

display(
    monthly_features_df[
        [
            "month",
            "calendar_year",
            "calendar_month",
            "calendar_quarter",
            "days_in_month",
            "is_december",
            "is_february",
        ]
    ].tail(15)
)

Calendar feature engineering: PASSED
Monthly observations: 55


,month,calendar_year,calendar_month,calendar_quarter,days_in_month,is_december,is_february
40,2016-05-01,2016,5,2,31,0,0
41,2016-06-01,2016,6,2,30,0,0
42,2016-07-01,2016,7,3,31,0,0
43,2016-08-01,2016,8,3,31,0,0
44,2016-09-01,2016,9,3,30,0,0
45,2016-10-01,2016,10,4,31,0,0
46,2016-11-01,2016,11,4,30,0,0
47,2016-12-01,2016,12,4,31,1,0
48,2017-01-01,2017,1,1,31,0,0
49,2017-02-01,2017,2,1,28,0,1


## Step 5 — Calendar Feature Distribution Validation

Validate the number of historical observations for each
calendar month and confirm the December and February flags.

The 2017 dataset contains only January through July,
so calendar-month observation counts will not be equal.

In [10]:
# Calendar-month observation distribution
calendar_distribution_df = (
    monthly_features_df
    .groupby("calendar_month", as_index=False)
    .agg(
        months_observed=("month", "size"),
        december_flags=("is_december", "sum"),
        february_flags=("is_february", "sum")
    )
)

# Validation
assert len(calendar_distribution_df) == 12
assert calendar_distribution_df["months_observed"].sum() == 55

assert monthly_features_df["is_december"].sum() == 4
assert monthly_features_df["is_february"].sum() == 5

assert (
    calendar_distribution_df["december_flags"].sum() == 4
)

assert (
    calendar_distribution_df["february_flags"].sum() == 5
)

assert (
    monthly_features_df["is_december"]
    == (monthly_features_df["calendar_month"] == 12).astype(int)
).all()

assert (
    monthly_features_df["is_february"]
    == (monthly_features_df["calendar_month"] == 2).astype(int)
).all()

print("Calendar feature distribution validation: PASSED")
print("Total monthly observations:", len(monthly_features_df))
print("December observations:", monthly_features_df["is_december"].sum())
print("February observations:", monthly_features_df["is_february"].sum())

display(calendar_distribution_df)

Calendar feature distribution validation: PASSED
Total monthly observations: 55
December observations: 4
February observations: 5


,calendar_month,months_observed,december_flags,february_flags
0,1,5,0,0
1,2,5,0,5
2,3,5,0,0
3,4,5,0,0
4,5,5,0,0
5,6,5,0,0
6,7,5,0,0
7,8,4,0,0
8,9,4,0,0
9,10,4,0,0


## Step 6 — Final Calendar Feature Quality Check

Validate calendar feature completeness, consistency,
and chronological continuity.

Only calendar information known before the forecast
month will be considered for future modeling.

In [11]:
# Calendar features to validate
calendar_columns = [
    "calendar_year",
    "calendar_month",
    "calendar_quarter",
    "days_in_month",
    "is_december",
    "is_february",
]

# 1. Missing values
missing_values = monthly_features_df[calendar_columns].isna().sum()

# 2. Duplicate monthly dates
duplicate_months = monthly_features_df["month"].duplicated().sum()

# 3. Chronological continuity
expected_months = pd.date_range(
    start="2013-01-01",
    end="2017-07-01",
    freq="MS"
)

# 4. Calendar consistency
assert missing_values.sum() == 0
assert duplicate_months == 0

assert monthly_features_df["month"].tolist() == expected_months.tolist()

assert (
    monthly_features_df["calendar_year"]
    == monthly_features_df["month"].dt.year
).all()

assert (
    monthly_features_df["calendar_quarter"]
    == monthly_features_df["month"].dt.quarter
).all()

assert (
    monthly_features_df["days_in_month"]
    == monthly_features_df["month"].dt.days_in_month
).all()

# 5. Check indicator consistency
assert (
    monthly_features_df["is_december"]
    == monthly_features_df["month"].dt.month.eq(12).astype(int)
).all()

assert (
    monthly_features_df["is_february"]
    == monthly_features_df["month"].dt.month.eq(2).astype(int)
).all()

print("Final calendar feature quality check: PASSED")
print("Monthly observations:", len(monthly_features_df))
print("Calendar features validated:", len(calendar_columns))
print("Missing feature values:", missing_values.sum())
print("Duplicate months:", duplicate_months)
print("Missing calendar months:", len(expected_months.difference(monthly_features_df["month"])))

Final calendar feature quality check: PASSED
Monthly observations: 55
Calendar features validated: 6
Missing feature values: 0
Duplicate months: 0
Missing calendar months: 0


## Step 7 — Historical Promotion Lag Features

Create historical promotion features using only data
available at the forecast origin.

The features in this step are exploratory.
Their safe use in three-month rolling-origin forecasts
will be implemented separately.

In [12]:
# Ensure chronological order
monthly_features_df = (
    monthly_features_df
    .sort_values("month")
    .reset_index(drop=True)
)

# Previous month's recorded promotion share
monthly_features_df["promotion_share_lag_1"] = (
    monthly_features_df["promotion_record_share_pct"].shift(1)
)

# Previous 3 months' average recorded promotion share
# shift(1) excludes the current month's promotion activity
monthly_features_df["promotion_share_lag_3_avg"] = (
    monthly_features_df["promotion_record_share_pct"]
    .shift(1)
    .rolling(window=3, min_periods=3)
    .mean()
)

# Previous 3 months' total promoted item counts
monthly_features_df["promoted_items_lag_3_sum"] = (
    monthly_features_df["total_promoted_item_counts"]
    .shift(1)
    .rolling(window=3, min_periods=3)
    .sum()
)

# Validation
assert len(monthly_features_df) == 55
assert monthly_features_df["month"].is_unique

assert monthly_features_df["promotion_share_lag_1"].isna().sum() == 1
assert monthly_features_df["promotion_share_lag_3_avg"].isna().sum() == 3
assert monthly_features_df["promoted_items_lag_3_sum"].isna().sum() == 3

assert (
    monthly_features_df["promotion_share_lag_1"].iloc[1:].to_numpy()
    == monthly_features_df["promotion_record_share_pct"].iloc[:-1].to_numpy()
).all()

print("Historical promotion lag feature creation: PASSED")
print("Monthly observations:", len(monthly_features_df))

display(
    monthly_features_df[
        [
            "month",
            "promotion_coverage_status",
            "promotion_record_share_pct",
            "promotion_share_lag_1",
            "promotion_share_lag_3_avg",
            "promoted_items_lag_3_sum",
        ]
    ].tail(12)
)

Historical promotion lag feature creation: PASSED
Monthly observations: 55


,month,promotion_coverage_status,promotion_record_share_pct,promotion_share_lag_1,promotion_share_lag_3_avg,promoted_items_lag_3_sum
43,2016-08-01,RECORDED_PERIOD,36.437855,40.909091,40.526572,920040.0
44,2016-09-01,RECORDED_PERIOD,35.858586,36.437855,39.401220,821613.0
45,2016-10-01,RECORDED_PERIOD,40.559719,35.858586,37.735177,832558.0
46,2016-11-01,RECORDED_PERIOD,44.369622,40.559719,37.618720,938467.0
47,2016-12-01,RECORDED_PERIOD,47.723532,44.369622,40.262642,1105233.0
48,2017-01-01,RECORDED_PERIOD,46.098983,47.723532,44.217624,1326370.0
49,2017-02-01,RECORDED_PERIOD,47.272326,46.098983,46.064045,1319284.0
50,2017-03-01,RECORDED_PERIOD,43.037906,47.272326,47.031614,1178180.0
51,2017-04-01,RECORDED_PERIOD,45.241302,43.037906,45.469738,999973.0
52,2017-05-01,RECORDED_PERIOD,49.187213,45.241302,45.183845,1007587.0


## Step 8 — Historical Promotion Lag Validation

Independently recalculate lag features from historical
monthly promotion records.

Confirm that the current month's promotion activity
is excluded from each lag feature.

In [13]:
# Select July 2017 for independent validation
target_month = pd.Timestamp("2017-07-01")

target_row = monthly_features_df.loc[
    monthly_features_df["month"] == target_month
].iloc[0]

# June 2017: previous month
june_row = monthly_features_df.loc[
    monthly_features_df["month"] == pd.Timestamp("2017-06-01")
].iloc[0]

# April–June 2017: previous three months
historical_3_months = monthly_features_df.loc[
    monthly_features_df["month"].between(
        "2017-04-01",
        "2017-06-01"
    )
]

expected_lag_1 = june_row["promotion_record_share_pct"]

expected_lag_3_avg = (
    historical_3_months["promotion_record_share_pct"].mean()
)

expected_promoted_items_sum = (
    historical_3_months["total_promoted_item_counts"].sum()
)

# Independent validation
assert len(historical_3_months) == 3

assert np.isclose(
    target_row["promotion_share_lag_1"],
    expected_lag_1
)

assert np.isclose(
    target_row["promotion_share_lag_3_avg"],
    expected_lag_3_avg
)

assert np.isclose(
    target_row["promoted_items_lag_3_sum"],
    expected_promoted_items_sum
)

print("Historical promotion lag validation: PASSED")
print("Target month:", target_month.date())
print("Previous month promotion share:", round(expected_lag_1, 4))
print("Previous 3-month average share:", round(expected_lag_3_avg, 4))
print("Previous 3-month promoted item count:", int(expected_promoted_items_sum))

display(
    historical_3_months[
        [
            "month",
            "promotion_record_share_pct",
            "total_promoted_item_counts"
        ]
    ]
)

Historical promotion lag validation: PASSED
Target month: 2017-07-01
Previous month promotion share: 44.3771
Previous 3-month average share: 46.2685
Previous 3-month promoted item count: 1180270


,month,promotion_record_share_pct,total_promoted_item_counts
51,2017-04-01,45.241302,371042
52,2017-05-01,49.187213,414646
53,2017-06-01,44.377104,394582


## Step 9 — Forecast-Origin-Safe Promotion Features

Construct historical promotion predictors separately for each
rolling forecast origin.

For a three-month forecast, all three target months use
promotion information available at the same forecast origin.

Do not use actual promotion data from validation months.

In [14]:
def build_origin_promotion_features(history_df):
    """
    Build promotion features using only data available
    at the forecast origin.

    history_df must contain historical months only.
    """
    history = history_df.sort_values("month").copy()

    assert len(history) >= 3
    assert history["month"].is_unique

    latest_3 = history.tail(3)

    return {
        "origin_month": history["month"].iloc[-1],

        "origin_promotion_share_lag_1":
            history["promotion_record_share_pct"].iloc[-1],

        "origin_promotion_share_3m_avg":
            latest_3["promotion_record_share_pct"].mean(),

        "origin_promoted_items_3m_sum":
            latest_3["total_promoted_item_counts"].sum(),

        "origin_promotion_coverage_complete":
            latest_3["promotion_coverage_status"]
            .eq("RECORDED_PERIOD").all(),
    }


# Test using April 2017 as the forecast origin
forecast_origin = pd.Timestamp("2017-04-01")

origin_history = monthly_features_df.loc[
    monthly_features_df["month"] <= forecast_origin
].copy()

origin_features = build_origin_promotion_features(
    origin_history
)

# Validation
assert origin_features["origin_month"] == forecast_origin

assert np.isclose(
    origin_features["origin_promotion_share_lag_1"],
    45.241302,
    atol=0.000001
)

assert origin_features["origin_promotion_coverage_complete"]

print("Forecast-origin-safe promotion feature test: PASSED")

display(pd.DataFrame([origin_features]))

Forecast-origin-safe promotion feature test: PASSED


,origin_month,origin_promotion_share_lag_1,origin_promotion_share_3m_avg,origin_promoted_items_3m_sum,origin_promotion_coverage_complete
0,2017-04-01,45.241302,45.183845,1007587,True


## Step 10 — Rolling-Origin Promotion Feature Construction

Construct historical promotion features for each
rolling-origin forecasting window.

Each three-month forecast uses the same promotion
information available at its forecast origin.

Validate the 17 origins, 51 forecast cases,
and absence of future promotion information.

In [15]:
# Load the existing 51 matched forecast cases
matched_forecasts_df = pd.read_csv(
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_evaluation"
    / "seven_model_matched_forecasts.csv"
)

print("Matched forecast rows:", len(matched_forecasts_df))
print("Columns:", matched_forecasts_df.columns.tolist())

display(matched_forecasts_df.head(3))

Matched forecast rows: 51
Columns: ['window_id', 'train_end', 'month', 'total_sales', 'naive_forecast', 'seasonal_naive_forecast', 'blended_forecast', 'ets_forecast', 'damped_ets_forecast', 'seasonal_index_forecast', 'trend_adjusted_forecast']


,window_id,train_end,month,total_sales,naive_forecast,seasonal_naive_forecast,blended_forecast,ets_forecast,damped_ets_forecast,seasonal_index_forecast,trend_adjusted_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,2.724398e+07,1.489692e+07,2.107045e+07,2.346665e+07,2.262673e+07,1.812011e+07,2.083680e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,1.374240e+07,2.049319e+07,1.930117e+07,1.910118e+07,1.471832e+07,1.692500e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,1.559861e+07,2.142129e+07,2.421934e+07,2.330344e+07,1.954081e+07,2.247050e+07


## Step 11 — Build Rolling-Origin Promotion Features

For each forecast window, calculate promotion features
using historical observations through train_end only.

Assign the same historical promotion features to all
three forecast months in that window.

Validate forecast timing and prevent future-data leakage.

In [16]:
# Prepare forecast dates
matched_forecasts_df["train_end"] = pd.to_datetime(
    matched_forecasts_df["train_end"]
)

matched_forecasts_df["month"] = pd.to_datetime(
    matched_forecasts_df["month"]
)

# Construct one feature record per forecast case
feature_records = []

for row in matched_forecasts_df.itertuples(index=False):

    # Historical data available at this forecast origin
    history = monthly_features_df.loc[
        monthly_features_df["month"] <= row.train_end
    ]

    # Reuse our validated function
    features = build_origin_promotion_features(history)

    feature_records.append({
        "window_id": row.window_id,
        "train_end": row.train_end,
        "forecast_month": row.month,
        "forecast_horizon": (
            (row.month.year - row.train_end.year) * 12
            + row.month.month - row.train_end.month
        ),
        **features,
    })

rolling_promotion_features_df = pd.DataFrame(feature_records)

# Validate forecast design
assert len(rolling_promotion_features_df) == 51
assert rolling_promotion_features_df["window_id"].nunique() == 17

assert (
    rolling_promotion_features_df.groupby("window_id").size() == 3
).all()

assert rolling_promotion_features_df["forecast_horizon"].between(
    1, 3
).all()

assert (
    rolling_promotion_features_df["origin_month"]
    == rolling_promotion_features_df["train_end"]
).all()

assert (
    rolling_promotion_features_df["forecast_month"]
    > rolling_promotion_features_df["origin_month"]
).all()

assert (
    rolling_promotion_features_df[
        "origin_promotion_coverage_complete"
    ]
).all()

print("Rolling-origin promotion feature construction: PASSED")
print("Forecast windows:",
      rolling_promotion_features_df["window_id"].nunique())
print("Forecast cases:", len(rolling_promotion_features_df))
print(
    "Forecast horizons:",
    sorted(rolling_promotion_features_df["forecast_horizon"].unique())
)

display(rolling_promotion_features_df.head(6))

Rolling-origin promotion feature construction: PASSED
Forecast windows: 17
Forecast cases: 51
Forecast horizons: [np.int64(1), np.int64(2), np.int64(3)]


,window_id,train_end,forecast_month,forecast_horizon,origin_month,origin_promotion_share_lag_1,origin_promotion_share_3m_avg,origin_promoted_items_3m_sum,origin_promotion_coverage_complete
0,1,2015-12-01,2016-01-01,1,2015-12-01,22.343808,21.192511,376710,True
1,1,2015-12-01,2016-02-01,2,2015-12-01,22.343808,21.192511,376710,True
2,1,2015-12-01,2016-03-01,3,2015-12-01,22.343808,21.192511,376710,True
3,2,2016-01-01,2016-02-01,1,2016-01-01,20.531480,21.088121,381068,True
4,2,2016-01-01,2016-03-01,2,2016-01-01,20.531480,21.088121,381068,True
5,2,2016-01-01,2016-04-01,3,2016-01-01,20.531480,21.088121,381068,True


## Step 12 — Rolling-Origin Feature Leakage Validation

Verify that all three forecast horizons in each window
use identical historical promotion features.

Independently recalculate each window's features from
historical records available at the forecast origin.

Confirm that no future promotion observations are used.

In [17]:
# Historical promotion features to validate
promotion_feature_columns = [
    "origin_promotion_share_lag_1",
    "origin_promotion_share_3m_avg",
    "origin_promoted_items_3m_sum",
    "origin_promotion_coverage_complete",
]

# 1. All three horizons must share the same feature values
feature_consistency = (
    rolling_promotion_features_df
    .groupby("window_id")[promotion_feature_columns]
    .nunique(dropna=False)
)

assert feature_consistency.eq(1).all().all()

# 2. Independently validate every forecast window
validated_windows = 0

for window_id, group in rolling_promotion_features_df.groupby(
    "window_id"
):
    origin = group["train_end"].iloc[0]

    # Reconstruct historical data available at the origin
    historical_data = monthly_features_df.loc[
        monthly_features_df["month"] <= origin
    ].sort_values("month")

    latest_three = historical_data.tail(3)

    assert len(latest_three) == 3
    assert historical_data["month"].max() == origin

    expected_latest_share = (
        historical_data["promotion_record_share_pct"].iloc[-1]
    )

    expected_average_share = (
        latest_three["promotion_record_share_pct"].mean()
    )

    expected_promoted_items = (
        latest_three["total_promoted_item_counts"].sum()
    )

    expected_coverage = (
        latest_three["promotion_coverage_status"]
        .eq("RECORDED_PERIOD")
        .all()
    )

    assert np.isclose(
        group["origin_promotion_share_lag_1"].iloc[0],
        expected_latest_share
    )

    assert np.isclose(
        group["origin_promotion_share_3m_avg"].iloc[0],
        expected_average_share
    )

    assert np.isclose(
        group["origin_promoted_items_3m_sum"].iloc[0],
        expected_promoted_items
    )

    assert (
        group["origin_promotion_coverage_complete"].iloc[0]
        == expected_coverage
    )

    # Forecast targets must occur after the origin
    assert (group["forecast_month"] > origin).all()

    # Each window must contain horizons 1, 2, and 3
    assert set(group["forecast_horizon"]) == {1, 2, 3}

    validated_windows += 1

assert validated_windows == 17

print("Rolling-origin promotion leakage validation: PASSED")
print("Windows independently validated:", validated_windows)
print("Forecast cases validated:", len(rolling_promotion_features_df))
print("Windows with inconsistent promotion features:",
      int((~feature_consistency.eq(1).all(axis=1)).sum()))
print("Future promotion observations used: 0")

Rolling-origin promotion leakage validation: PASSED
Windows independently validated: 17
Forecast cases validated: 51
Windows with inconsistent promotion features: 0
Future promotion observations used: 0


## Step 13 — Export Validated Forecast Features

Export the 51 rolling-origin forecast cases with their
historical promotion features and advance-known calendar features.

Preserve the original forecast targets and benchmark predictions
for subsequent matched-case model evaluation.

In [18]:
# Create a copy of the validated rolling-origin features
forecast_features_export_df = rolling_promotion_features_df.copy()

# Add calendar features for the target forecast month
forecast_features_export_df["calendar_year"] = (
    forecast_features_export_df["forecast_month"].dt.year
)

forecast_features_export_df["calendar_month"] = (
    forecast_features_export_df["forecast_month"].dt.month
)

forecast_features_export_df["calendar_quarter"] = (
    forecast_features_export_df["forecast_month"].dt.quarter
)

forecast_features_export_df["days_in_month"] = (
    forecast_features_export_df["forecast_month"].dt.days_in_month
)

forecast_features_export_df["is_december"] = (
    forecast_features_export_df["calendar_month"].eq(12).astype(int)
)

forecast_features_export_df["is_february"] = (
    forecast_features_export_df["calendar_month"].eq(2).astype(int)
)

# Add actual sales and historical benchmark forecasts
forecast_features_export_df = forecast_features_export_df.merge(
    matched_forecasts_df[
        [
            "window_id",
            "train_end",
            "month",
            "total_sales",
            "blended_forecast",
            "ets_forecast",
        ]
    ],
    left_on=["window_id", "train_end", "forecast_month"],
    right_on=["window_id", "train_end", "month"],
    how="left",
    validate="one_to_one"
)

# Remove duplicate month column
forecast_features_export_df = forecast_features_export_df.drop(
    columns=["month"]
)

# Validate the exported dataset
assert len(forecast_features_export_df) == 51

assert forecast_features_export_df[
    ["window_id", "forecast_month"]
].duplicated().sum() == 0

assert forecast_features_export_df[
    [
        "total_sales",
        "blended_forecast",
        "ets_forecast",
    ]
].notna().all().all()

assert forecast_features_export_df[
    "origin_promotion_coverage_complete"
].all()

assert (
    forecast_features_export_df["forecast_month"]
    > forecast_features_export_df["train_end"]
).all()

# Save the report
report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_engineering"
)

report_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    report_dir
    / "rolling_origin_calendar_promotion_features.csv"
)

forecast_features_export_df.to_csv(
    output_path,
    index=False
)

# Read the saved file to verify successful export
saved_features_df = pd.read_csv(output_path)

assert len(saved_features_df) == 51
assert saved_features_df.shape[1] == forecast_features_export_df.shape[1]

print("Rolling-origin feature export: PASSED")
print("Report:", output_path.relative_to(PROJECT_ROOT))
print("Forecast windows:", forecast_features_export_df["window_id"].nunique())
print("Forecast cases:", len(saved_features_df))
print("Columns exported:", len(saved_features_df.columns))

display(forecast_features_export_df.head(6))

Rolling-origin feature export: PASSED
Report: reports\forecasting_feature_engineering\rolling_origin_calendar_promotion_features.csv
Forecast windows: 17
Forecast cases: 51
Columns exported: 18


,window_id,train_end,forecast_month,forecast_horizon,origin_month,origin_promotion_share_lag_1,origin_promotion_share_3m_avg,origin_promoted_items_3m_sum,origin_promotion_coverage_complete,calendar_year,calendar_month,calendar_quarter,days_in_month,is_december,is_february,total_sales,blended_forecast,ets_forecast
0,1,2015-12-01,2016-01-01,1,2015-12-01,22.343808,21.192511,376710,True,2016,1,1,31,0,0,2.397780e+07,2.107045e+07,2.346665e+07
1,1,2015-12-01,2016-02-01,2,2015-12-01,22.343808,21.192511,376710,True,2016,2,1,29,0,1,2.194741e+07,2.049319e+07,1.930117e+07
2,1,2015-12-01,2016-03-01,3,2015-12-01,22.343808,21.192511,376710,True,2016,3,1,31,0,0,2.313178e+07,2.142129e+07,2.421934e+07
3,2,2016-01-01,2016-02-01,1,2016-01-01,20.531480,21.088121,381068,True,2016,2,1,29,0,1,2.194741e+07,1.886010e+07,1.936486e+07
4,2,2016-01-01,2016-03-01,2,2016-01-01,20.531480,21.088121,381068,True,2016,3,1,31,0,0,2.313178e+07,1.978821e+07,2.428877e+07
5,2,2016-01-01,2016-04-01,3,2016-01-01,20.531480,21.088121,381068,True,2016,4,2,30,0,0,2.596303e+07,1.946644e+07,2.006642e+07
